# 02 · 必做：Controlled Experiment / 学习率对照实验
预计 50–60 分钟，包括理解、运行、比较和讨论。Behavior Cloning（行为克隆）通过专家动作训练策略。只改变学习率，固定数据与完整轨迹划分、256×3 MLP 结构、随机种子17、50轮、batch size及归一化。学习率预先固定为 1e-3 和 3e-4；必须完成两组，不能依据测试分数挑选模型或种子。

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
GROUP = os.environ.get("LAB_GROUP", "group_01")  # 教师为每组分配；只改自己的组
RUN = workspace(GROUP)
from manipulation_lab.teaching import read_json, markdown_table
settings = read_json(ROOT / "configs/bc_student.json")
design = read_json(ROOT / "configs/controlled_experiment.json")
print(settings)
print(design)


核心更新逻辑如下；真正训练由后面的独立脚本调用官方 robomimic BC。专家动作是标签，网络预测不参与产生这些标签。验证集由完整轨迹组成，用于选择 checkpoint；测试集不参与更新或模型选择。
```python
pred_action = policy(obs)
loss = criterion(pred_action, expert_action)
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

In [ ]:
import h5py, torch
from manipulation_lab.common import policy_observation
from manipulation_lab.policy import make_config, make_model
from manipulation_lab.dataset import inspect
info = inspect(DATA)
config = make_config(settings)
example_model = make_model(config, info["policy_observation_shapes"], torch.device("cpu"))
print(example_model.nets["policy"])
with h5py.File(DATA, "r") as f:
    episode = info["splits"]["train"][0]
    print("Expert action labels (first 3):", f[f"data/{episode}/actions"][:3])
del example_model

两组使用完全相同的初始模型参数和训练顺序种子。只在验证集上选择各自的 checkpoint（保存的模型参数）；测试结果用于对照分析。下面会实际训练两次，并分别在相同标准5次＋位置偏移5次条件下评测。每个模型、曲线、配置和回合均保存，不覆盖原实验。

In [ ]:
# 无 GPU 时先将 device 明确改为 cpu；两组必须相同。不要改 seed、epochs 或结构。
settings["device"] = "cuda"
EXPERIMENT = next_stage(RUN, "experiment")
BASE_CONFIG = RUN / (EXPERIMENT.name + "_base_config.json")
write_json(BASE_CONFIG, settings)
command("controlled_experiment.py", "--config", BASE_CONFIG,
        "--design", ROOT / "configs/controlled_experiment.json",
        "--evaluation-config", ROOT / "configs/evaluation.yaml", "--output", EXPERIMENT, timeout=14400)
experiment = read_json(EXPERIMENT / "experiment.json")
assert experiment["complete"] and all(experiment["checks"].values())
print("Controlled checks:", experiment["checks"])
for variant, paths in experiment["variants"].items():
    checkpoint = EXPERIMENT / paths["training"] / "student_policy.pth"
    print(variant, "checkpoint:", checkpoint, "SHA256:", paths["checkpoint_sha256"])


In [ ]:
display(Image(filename=str(EXPERIMENT / "learning_rate_curves.png")))
display(Markdown(markdown_table(experiment["comparison"])))
for variant, paths in experiment["variants"].items():
    display(Markdown("### " + variant + " · LR=" + str(paths["learning_rate"])))
    display(Image(filename=str(EXPERIMENT / paths["training"] / "training_curve.png")))
print("原始训练 CSV、两份 checkpoint、20 次仿真记录位于：", EXPERIMENT)


讨论：较低训练损失是否对应较高成功率？学习率对收敛速度和验证误差有什么影响？每种条件只有5次，不足以证明细微优劣。后续固定分析 lr_a，不能看完测试再挑选更高分模型。Notebook 03 将使用这两组已完成的同条件结果绘图和逐案例配对；不重复仿真。